In [1]:
!git clone https://github.com/huzaifa-ai-ml/movie-recommendation-system.git

Cloning into 'movie-recommendation-system'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 44 (delta 16), reused 27 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 37.80 MiB | 13.34 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (10/10), done.


In [2]:
import os

os.chdir('/content/movie-recommendation-system')

print("Project directory:", os.getcwd())
print("Dataset files:", os.listdir('dataset'))

Project directory: /content/movie-recommendation-system
Dataset files: ['processed_movies.csv', 'movies.csv']


In [3]:
# 1. Import Libraries

import pandas as pd
import pickle

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
# 2. Set Project Paths

ROOT_DIR = Path.cwd()

DATASET_DIR = ROOT_DIR / 'dataset'
MODELS_DIR = ROOT_DIR / 'models'

MODELS_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
# 3. Load Processed Dataset

df = pd.read_csv(DATASET_DIR / 'processed_movies.csv')

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (45447, 7)


In [6]:
print("Missing tags:", df['tags'].isna().sum())
print("Empty tags:", (df['tags'].fillna('').str.strip() == '').sum())

Missing tags: 260
Empty tags: 260


In [7]:
# Handle missing and empty tags
df['tags'] = df['tags'].fillna('').astype(str)

# Remove rows with empty tags
df = df[df['tags'].str.strip() != ''].reset_index(drop=True)

# Save Recommendation Dataset
df.to_csv(DATASET_DIR / 'recommendation_movies.csv', index=False)

print("Dataset shape after removing empty tags:", df.shape)
print("Missing tags:", df['tags'].isna().sum())


Dataset shape after removing empty tags: (45187, 7)
Missing tags: 0


In [8]:
# 4. Create Movie Title Index Mapping

indices = pd.Series(
    df.index,
    index=df['title']
).drop_duplicates()

print("Movie index mapping created!")

Movie index mapping created!


In [9]:
# 5. Initialize TF-IDF Vectorizer

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words='english'
)

In [10]:
# 6. Generate TF-IDF Matrix

tfidf_matrix = tfidf.fit_transform(df['tags'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))

TF-IDF Matrix Shape: (45187, 50000)
Number of features: 50000


In [11]:
# 7. Save Model Artifacts

with open(MODELS_DIR / 'tfidf_matrix.pkl', 'wb') as file:
    pickle.dump(tfidf_matrix, file)

with open(MODELS_DIR / 'indices.pkl', 'wb') as file:
    pickle.dump(indices, file)

with open(MODELS_DIR / 'tfidf.pkl', 'wb') as file:
    pickle.dump(tfidf, file)

print("\nTF-IDF artifacts saved successfully!")

print("\nGenerated files:")
for file in MODELS_DIR.iterdir():
    print(file.name)


TF-IDF artifacts saved successfully!

Generated files:
tfidf_matrix.pkl
tfidf.pkl
indices.pkl
